In [2]:
import cme.decision_models.confidence_accumulation as ca
import numpy as np
import scipy.stats as stats
import pandas as pd
import seaborn as sns
import numpyro as npy
import numpyro.distributions as dist
import jax

In [3]:
rng = jax.random.key(1)
rng

Array((), dtype=key<fry>) overlaying:
[0 1]

In [4]:
jax.random.split(rng)

Array((2,), dtype=key<fry>) overlaying:
[[2441914641 1384938218]
 [3819641963 2025898573]]

In [5]:
max_samples, max_RT = 400, 40

In [6]:
n_states, start_width, delta, measurement_prob,  = 101, 11, 0.5, 0.8
I, J = 20, 100

In [7]:
threshold, measurement_prob, mu, sigma = 1, 1, np.asarray([[1]]), np.asarray([[2]])
model_type = "Markov"
transition_type="TIMESTEP"#"RT"

In [8]:
#t_s = stats.uniform(delta,max_RT/delta).rvs((I,max_samples))
t_s = dist.Uniform(delta, max_RT/delta).sample(key=(rng := jax.random.split(rng)[0]), sample_shape=(I,max_samples))
t_s.shape

(20, 400)

In [9]:
t_s

Array([[34.69442518, 61.7658194 ,  0.62858804, ..., 73.07466942,
        25.30199071, 63.13220686],
       [45.73041269, 17.10251804, 26.64940844, ..., 58.0509323 ,
        70.77729065, 70.93219106],
       [76.14301467, 44.26267266, 20.22450108, ..., 14.3577864 ,
        35.46746188,  5.49930964],
       ...,
       [66.53874692, 26.9696467 ,  1.34027883, ...,  9.23885716,
        40.42430122, 50.81786815],
       [ 6.42067279, 16.07337827, 24.72165412, ..., 10.43910673,
        38.33478562,  4.14355251],
       [29.33508139, 78.22842921, 58.26652618, ..., 44.76425108,
        69.00048595, 46.59965439]], dtype=float64)

In [10]:
intensity_matrix = ca.get_intensity_matrix(n_states, mu, sigma, model_type=model_type)
phi_0 = ca._get_initial_state(n_states, start_width,model_type=model_type, prior_type="Centered")
intensity_matrix, phi_0

(Array([[[[-2. ,  0.5,  0. , ...,  0. ,  0. ,  0. ],
          [ 2. , -2. ,  0.5, ...,  0. ,  0. ,  0. ],
          [ 0. ,  1.5, -2. , ...,  0. ,  0. ,  0. ],
          ...,
          [ 0. ,  0. ,  0. , ..., -2. ,  0.5,  0. ],
          [ 0. ,  0. ,  0. , ...,  1.5, -2. ,  2. ],
          [ 0. ,  0. ,  0. , ...,  0. ,  1.5, -2. ]]]], dtype=float64),
 Array([[[[0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],
          [0.        ],


In [11]:
Mc, Mw, Mn = ca._get_measurement_matrix(n_states, threshold, prob=measurement_prob, model_type = model_type)
Mc

Array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], dtype=float64)

In [12]:
t = t_s #np.asarray([[4.5]]) #
phi_t = ca.perform_state_transition(intensity_matrix=intensity_matrix, RT_s=t, RA_s = None, delta=delta, 
Mc = Mc, Mn = Mn, Mw = Mw, phi_0=phi_0, 
transition_type=transition_type, likelihood_type="SINGLE")
phi_t.shape

(1, 400, 101, 1)

In [13]:
phi_t

Array([[[[4.39713513e-25],
         [2.89243488e-24],
         [1.11385604e-23],
         ...,
         [1.57515290e-02],
         [1.48097519e-02],
         [6.53192770e-03]],

        [[1.56268481e-25],
         [9.72207458e-25],
         [3.21150637e-24],
         ...,
         [1.69625997e-02],
         [1.96571088e-02],
         [9.17905537e-03]],

        [[5.20955552e-70],
         [2.27823410e-68],
         [9.61362924e-67],
         ...,
         [3.31625261e-46],
         [2.36310032e-47],
         [1.61782998e-48]],

        ...,

        [[2.48209142e-26],
         [1.53174632e-25],
         [4.93929989e-25],
         ...,
         [3.92231939e-03],
         [4.72411599e-03],
         [2.22679939e-03]],

        [[3.44184938e-26],
         [2.40930738e-25],
         [1.07852203e-24],
         ...,
         [9.12016391e-04],
         [7.14223349e-04],
         [2.96866990e-04]],

        [[1.26277396e-25],
         [7.84597692e-25],
         [2.58182667e-24],
         ...,
 

In [14]:
phi_t.sum(axis=-2, keepdims=True).sum()

Array(253.02874692, dtype=float64)

In [15]:
[phi_i_j.squeeze().sum() for phi_i in phi_t for phi_i_j in phi_i ]

[Array(0.95499575, dtype=float64),
 Array(0.15033236, dtype=float64),
 Array(1., dtype=float64),
 Array(1., dtype=float64),
 Array(0.08270249, dtype=float64),
 Array(0.99797368, dtype=float64),
 Array(1., dtype=float64),
 Array(0.52722183, dtype=float64),
 Array(0.08867134, dtype=float64),
 Array(0.99997965, dtype=float64),
 Array(0.88400136, dtype=float64),
 Array(0.02263736, dtype=float64),
 Array(0.8020417, dtype=float64),
 Array(0.5079406, dtype=float64),
 Array(1., dtype=float64),
 Array(0.97102968, dtype=float64),
 Array(0.26824877, dtype=float64),
 Array(0.99176733, dtype=float64),
 Array(0.1241167, dtype=float64),
 Array(0.15033236, dtype=float64),
 Array(0.45111723, dtype=float64),
 Array(0.93346856, dtype=float64),
 Array(0.99797368, dtype=float64),
 Array(0.34525107, dtype=float64),
 Array(0.9058873, dtype=float64),
 Array(0.1241167, dtype=float64),
 Array(0.99976387, dtype=float64),
 Array(0.92496666, dtype=float64),
 Array(0.45111723, dtype=float64),
 Array(0.88400136, dty

In [16]:
phi_t[0,0,:,0]

Array([4.39713513e-25, 2.89243488e-24, 1.11385604e-23, 3.85959409e-23,
       1.28951487e-22, 4.22679196e-22, 1.36482897e-21, 4.34540925e-21,
       1.36437634e-20, 4.22446987e-20, 1.28975794e-19, 3.88240552e-19,
       1.15214769e-18, 3.37045099e-18, 9.71849323e-18, 2.76184893e-17,
       7.73482667e-17, 2.13457416e-16, 5.80420794e-16, 1.55491844e-15,
       4.10362780e-15, 1.06681161e-14, 2.73169165e-14, 6.88914991e-14,
       1.71102610e-13, 4.18478296e-13, 1.00782271e-12, 2.38979816e-12,
       5.57926060e-12, 1.28234363e-11, 2.90148394e-11, 6.46250996e-11,
       1.41685915e-10, 3.05757746e-10, 6.49436125e-10, 1.35765190e-09,
       2.79331751e-09, 5.65615164e-09, 1.12715022e-08, 2.21052761e-08,
       4.26637597e-08, 8.10343802e-08, 1.51470080e-07, 2.78634248e-07,
       5.04425954e-07, 8.98717699e-07, 1.57587676e-06, 2.71961197e-06,
       4.61947399e-06, 7.72318910e-06, 1.27098241e-05, 2.05894047e-05,
       3.28348121e-05, 5.15512623e-05, 7.96870966e-05, 1.21286609e-04,
      

stats.multinomial(n=1, p=phi_t[0,0,:,0]).rvs()

In [17]:
phi_t

Array([[[[4.39713513e-25],
         [2.89243488e-24],
         [1.11385604e-23],
         ...,
         [1.57515290e-02],
         [1.48097519e-02],
         [6.53192770e-03]],

        [[1.56268481e-25],
         [9.72207458e-25],
         [3.21150637e-24],
         ...,
         [1.69625997e-02],
         [1.96571088e-02],
         [9.17905537e-03]],

        [[5.20955552e-70],
         [2.27823410e-68],
         [9.61362924e-67],
         ...,
         [3.31625261e-46],
         [2.36310032e-47],
         [1.61782998e-48]],

        ...,

        [[2.48209142e-26],
         [1.53174632e-25],
         [4.93929989e-25],
         ...,
         [3.92231939e-03],
         [4.72411599e-03],
         [2.22679939e-03]],

        [[3.44184938e-26],
         [2.40930738e-25],
         [1.07852203e-24],
         ...,
         [9.12016391e-04],
         [7.14223349e-04],
         [2.96866990e-04]],

        [[1.26277396e-25],
         [7.84597692e-25],
         [2.58182667e-24],
         ...,
 

def test(a,d):
    print(a.shape)
    return a

np.apply_over_axes(test, phi_t, axes=[0]).shape
#np.apply_along_axis(test, -2, phi_t).shape

In [18]:
states_t = dist.Multinomial(total_count=1, probs=phi_t[...,0]).sample(key=(rng:=jax.random.split(rng)[0]))
states_t.shape

(1, 400, 101)

[stats.multinomial(n=1, p=phi_i_j.squeeze()).rvs() for phi_i in phi_t for phi_i_j in phi_i ]

states_t1 = np.array([stats.multinomial(n=1, p=phi_i_j.squeeze()).rvs() for phi_i in phi_t for phi_i_j in phi_i ])
states_t1.shape

In [19]:
np.argwhere(states_t)

array([[  0,   0,  91],
       [  0,   1,  96],
       [  0,   2,  51],
       ...,
       [  0, 397,  99],
       [  0, 398,  82],
       [  0, 399,  95]])

In [20]:
np.argwhere(states_t).T

array([[  0,   0,   0, ...,   0,   0,   0],
       [  0,   1,   2, ..., 397, 398, 399],
       [ 91,  96,  51, ...,  99,  82,  95]])

In [21]:
state_final = np.argwhere(states_t)
state_final

array([[  0,   0,  91],
       [  0,   1,  96],
       [  0,   2,  51],
       ...,
       [  0, 397,  99],
       [  0, 398,  82],
       [  0, 399,  95]])

In [22]:
n_states - threshold

100

In [23]:
t_s.shape

(20, 400)

In [24]:
state_final >= n_states - threshold - 1

array([[False, False, False],
       [False, False, False],
       [False, False, False],
       ...,
       [False,  True,  True],
       [False,  True, False],
       [False,  True, False]])

np.where((state_final < threshold - 1) | (state_final >= n_states - threshold - 1))

In [25]:
RA = np.select([
    state_final[:,[-1]] < threshold - 1,
    state_final[:,[-1]] >= n_states - threshold - 1
], [0,1], default = np.nan)#[:,-1]
RA.shape

(400, 1)

In [26]:
t_s.flatten()[:,None].shape

(8000, 1)

In [27]:
state_final.shape

(400, 3)

In [28]:
(t_s.flatten()[:,None]).shape

(8000, 1)

In [29]:
Response = np.hstack((state_final, RA, t_s.flatten()[:,None]*delta))
Response

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 400 and the array at index 2 has size 8000

In [ ]:
df_res = pd.DataFrame(Response, columns=["I", "J", "state", "RA", "RT"])
df_res

In [ ]:
df_res.dropna().groupby(["I"]).count()

In [ ]:
df_res = df_res.dropna()
get_id = (df_res.groupby(["I"]).count() < J)[["J"]].query("J == True").index.values
get_id

In [ ]:
df_sample = df_res.groupby(["I"]).nth(slice(None, J)).drop("J", axis=1)
df_sample

In [ ]:
df_sample.groupby(["I"]).count()

In [ ]:
df_sample.RT.plot.kde()

In [ ]:
df_sample.RA.plot()